Generate graphics for the paper

In [1]:
import pandas as pd

df = pd.read_csv("../dataset/qgenllm-updated-filtered-min_q.csv", sep=";")
print("ONTOLOGIES")
print(df.ontology.unique())

ONTOLOGIES
['AWO' 'BioTop' 'CopyrightAll' 'Cultural-On' 'Pizza' 'Stuff' 'olia']


In [17]:
# print number of axioms per ontology
print("NUMBER OF AXIOMS PER ONTOLOGY")
print(df.groupby("ontology").size())
print(len(df))

NUMBER OF AXIOMS PER ONTOLOGY
ontology
AWO               1153
BioTop             149
CopyrightAll       293
Cultural-On      22668
Pizza            62701
Stuff           340412
olia                68
dtype: int64
427444


In [3]:
# unique number of question_type per ontology
print("UNIQUE NUMBER OF QUESTION TYPES PER ONTOLOGY")
print(df.groupby("ontology")["question_type"].nunique())

UNIQUE NUMBER OF QUESTION TYPES PER ONTOLOGY
ontology
AWO             8
BioTop          1
CopyrightAll    6
Cultural-On     8
Pizza           8
Stuff           8
olia            4
Name: question_type, dtype: int64


In [9]:
# For each ontology, find the same question associated with different question_types
print("SAME QUESTION WITH DIFFERENT QUESTION TYPES PER ONTOLOGY")
for ontology in df.ontology.unique():
    df_ont = df[df.ontology == ontology]
    question_counts = df_ont.groupby("question")["question_type"].nunique()
    diff_question_types = question_counts[question_counts > 1]
    if not diff_question_types.empty:
        print(f"Ontology: {ontology}")
        for question in diff_question_types.index:
            types = df_ont[df_ont.question == question]["question_type"].unique()
            quests = df_ont[df_ont.question == question][["question"]].values.flatten().tolist()
            print(f"  Question: {question} | Types: {types}")
            for q in quests:
                print(f"    {q}")
            # Print the questions
            

SAME QUESTION WITH DIFFERENT QUESTION TYPES PER ONTOLOGY
Ontology: AWO
  Question: A bumble bee participates in aviating. True or false? | Types: ['op_Yes-No-2-part-1-rel' 'op_Yes-No-2-part-1-rel+1-quant-some']
    A bumble bee participates in aviating. True or false?
    A bumble bee participates in aviating. True or false?
  Question: A bumble bee participates in flying. True or false? | Types: ['op_Yes-No-2-part-1-rel' 'op_Yes-No-2-part-1-rel+1-quant-some']
    A bumble bee participates in flying. True or false?
    A bumble bee participates in flying. True or false?
  Question: A bumble bee participates in hibernating. True or false? | Types: ['op_Yes-No-2-part-1-rel' 'op_Yes-No-2-part-1-rel+1-quant-some']
    A bumble bee participates in hibernating. True or false?
    A bumble bee participates in hibernating. True or false?
  Question: A bumble bee participates in holing up. True or false? | Types: ['op_Yes-No-2-part-1-rel' 'op_Yes-No-2-part-1-rel+1-quant-some']
    A bumble bee 

In [16]:
# Produce a latex table with one example per question type. Incude the ontology, the axiom, the question type and the question.

sub_df = df.groupby(["ontology", "question_type"]).sample(3, random_state=3).reset_index()
q_exs = sub_df.groupby("question_type").sample(1).reset_index()
latex_table = q_exs[["axiom",  "question", "question_type", "ontology"]].to_latex(index=False)
print("LATEX TABLE WITH ONE EXAMPLE PER QUESTION TYPE")
print(latex_table)

LATEX TABLE WITH ONE EXAMPLE PER QUESTION TYPE
\begin{tabular}{llll}
\toprule
axiom & question & question_type & ontology \\
\midrule
ProcessQuality SubclassOf(Particular) & What is a process quality? & Definition & BioTop \\
Perform SubclassOf(result only Performance) & What does a perform result? & op_What-1-part-1-rel & CopyrightAll \\
Gel SubclassOf(hasQuale only Solid),Gel SubclassOf(DispersionColloid) & Which dispersion colloid has a quale that is being a solid? & op_What-2-part-1-rel & Stuff \\
Use SubclassOf(Action),Use SubclassOf(agent only LegalPerson) & Which disagree action has an agent that is only a legal person? & op_What-2-part-1-rel-quant-only & CopyrightAll \\
Warthog SubclassOf(eats some PlantParts),Warthog SubclassOf(Omnivore) & Which omnivore eats some plant parts? & op_What-2-part-1-rel-quant-some & AWO \\
OnionTopping SubclassOf(hasSpiciness only ValuePartition) & An onion topping has a spiciness that is a value partition. True or false? & op_Yes-No-2-part-1-rel 

Now we open the dataset used for generation (sampled N questions per question type)

In [18]:
import pandas as pd
import pyarrow.parquet as pq

in_filename = "../dataset/qgenllm-ds-seed-123-few-shots.parquet"
df = pq.read_table(in_filename).to_pandas()

seed = in_filename.split("seed-")[1].split("-few")[0]
print("seed is:", seed)

print(len(df))

seed is: 123
430


In [20]:
df[df.ontology == "BioTop"]["question_type"].value_counts()

question_type
Definition    10
Name: count, dtype: int64